# 2. Create bigWig tracks for IGV

Из таблицы CpG methylation сделать `bedGraph` и `bigWig` треки для IGV

Создадим несколько треков:

- beta-value methylation
- M-value
- coverage
- GC content
- CpG observed/expected

Для `bigWig` нужен файл `chrom.sizes`, а интервалы должны быть отсортированы по `chrom/start/end`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyBigWig
import pysam

sample = "MoPh7"

methylation_table = Path("../results/tables/MoPh7_cpg_methylation_values.tsv.gz")
chrom_sizes_file = Path("../../day1_HiC_practice/data/reference/chrom.sizes")
reference_fasta = Path("../../day1_HiC_practice/data/reference/T2T_human.fna")

tracks_dir = Path("../results/tracks")
tracks_dir.mkdir(parents=True, exist_ok=True)

In [2]:
def read_chrom_sizes(path):
    chrom_sizes = {}
    with open(path) as handle:
        for line in handle:
            chrom, size = line.rstrip().split("	")[:2]
            chrom_sizes[chrom] = int(size)
    return chrom_sizes

chrom_sizes = read_chrom_sizes(chrom_sizes_file)
list(chrom_sizes.items())[:5]

[('chr1', 248387328),
 ('chr2', 242696752),
 ('chr3', 201105948),
 ('chr4', 193574945),
 ('chr5', 182045439)]

### Читаем таблицу метилирования

В первом ноутбуке мы сохранили отфильтрованную таблицу. Для IGV-трека переведем координаты CpG в `0-based` интервалы длиной 1 bp

In [3]:
df = pd.read_csv(methylation_table, sep="	")

df["bw_start"] = (df["start"] - 1).clip(lower=0).astype(int)
df["bw_end"] = df["bw_start"] + 1

df = df[df["chrom"].isin(chrom_sizes)].copy()
df = df.sort_values(["chrom", "bw_start", "bw_end"])

# Если одинаковая CpG-позиция встретилась дважды, усредняем значения.
df = (
    df.groupby(["chrom", "bw_start", "bw_end"], as_index=False)
    .agg({"beta_value": "mean", "m_value": "mean", "coverage": "mean"})
)

df.head()

,chrom,bw_start,bw_end,beta_value,m_value,coverage
0,chr1,3388,3389,0.800000,1.321928,5.0
1,chr1,3408,3409,1.000000,2.584963,5.0
2,chr1,3416,3417,0.833333,1.584963,6.0
3,chr1,3422,3423,1.000000,3.000000,7.0
4,chr1,3426,3427,1.000000,3.169925,8.0


In [4]:
def write_bedgraph(table, value_col, output_path):
    bed = table[["chrom", "bw_start", "bw_end", value_col]].copy()
    bed = bed.replace([np.inf, -np.inf], np.nan).dropna()
    bed.to_csv(output_path, sep="	", header=False, index=False)
    return output_path


def write_bigwig(table, value_col, output_path):
    table = table[["chrom", "bw_start", "bw_end", value_col]].copy()
    table = table.replace([np.inf, -np.inf], np.nan).dropna()
    table = table.sort_values(["chrom", "bw_start", "bw_end"])

    bw = pyBigWig.open(str(output_path), "w")
    used_chroms = [chrom for chrom in chrom_sizes if chrom in set(table["chrom"])]
    bw.addHeader([(chrom, chrom_sizes[chrom]) for chrom in used_chroms])

    for chrom, part in table.groupby("chrom", sort=False):
        bw.addEntries(
            [chrom] * len(part),
            part["bw_start"].astype(int).tolist(),
            ends=part["bw_end"].astype(int).tolist(),
            values=part[value_col].astype(float).tolist(),
        )
    bw.close()
    return output_path

In [ ]:
tracks = {
    "beta_methylation": "beta_value",
    "m_value": "m_value",
    "coverage": "coverage",
}

created_tracks = []

for track_name, column in tracks.items():
    bedgraph_path = tracks_dir / f"{sample}_{track_name}.bedGraph"
    bigwig_path = tracks_dir / f"{sample}_{track_name}.bw"

    write_bedgraph(df, column, bedgraph_path)
    write_bigwig(df, column, bigwig_path)

    created_tracks.extend([bedgraph_path, bigwig_path])

created_tracks

### GC content и CpG observed/expected

Теперь сделаем простые треки CpG cjcnfdf. Делим анализируемый участок генома на бины по 100 bp и считаем:

```text
GC content = (G + C) / length
CpG observed/expected = count(CG) / (count(C) * count(G) / length)
```

Чтобы ноутбук запускался быстро, берем только те участки, которые есть в таблице метилирования. Для полного генома можно пройтись по всем хромосомам из `chrom.sizes`

In [8]:
bin_size = 100
fasta = pysam.FastaFile(str(reference_fasta))

regions = (
    df.groupby("chrom")
    .agg(start=("bw_start", "min"), end=("bw_end", "max"))
    .reset_index()
)

regions.head()

,chrom,start,end
0,chr1,3388,248383305
1,chr10,3873,134754994
2,chr11,1758,135125125
3,chr12,10051,133322344
4,chr13,2835,113563207


In [9]:
gc_rows = []

for _, row in regions.iterrows():
    chrom = row["chrom"]
    region_start = int(row["start"] // bin_size * bin_size)
    region_end = int(np.ceil(row["end"] / bin_size) * bin_size)
    region_end = min(region_end, chrom_sizes[chrom])

    for start in range(region_start, region_end, bin_size):
        end = min(start + bin_size, chrom_sizes[chrom])
        seq = fasta.fetch(chrom, start, end).upper()
        length = len(seq)
        if length == 0:
            continue

        c_count = seq.count("C")
        g_count = seq.count("G")
        cg_count = seq.count("CG")

        gc_content = (c_count + g_count) / length
        expected_cpg = (c_count * g_count) / length if length else 0
        cpg_oe = cg_count / expected_cpg if expected_cpg > 0 else np.nan

        gc_rows.append((chrom, start, end, gc_content, cpg_oe))

gc_df = pd.DataFrame(gc_rows, columns=["chrom", "bw_start", "bw_end", "gc_content", "cpg_obs_exp"])
gc_df.head()

,chrom,bw_start,bw_end,gc_content,cpg_obs_exp
0,chr1,3300,3400,0.70,0.925147
1,chr1,3400,3500,0.69,0.854701
2,chr1,3500,3600,0.65,0.473485
3,chr1,3600,3700,0.69,0.504202
4,chr1,3700,3800,0.64,0.392157


In [ ]:
for track_name, column in {"gc_content": "gc_content", "cpg_obs_exp": "cpg_obs_exp"}.items():
    bedgraph_path = tracks_dir / f"T2T_{track_name}_100bp.bedGraph"
    bigwig_path = tracks_dir / f"T2T_{track_name}_100bp.bw"

    write_bedgraph(gc_df, column, bedgraph_path)
    write_bigwig(gc_df, column, bigwig_path)

    created_tracks.extend([bedgraph_path, bigwig_path])


## IGV

Откройте наш референсный геном genome и подгрузите

- `results/tracks/MoPh7_beta_methylation.bw`
- `results/tracks/MoPh7_coverage.bw`
- `results/tracks/T2T_gc_content_100bp.bw`
- `results/tracks/T2T_cpg_obs_exp_100bp.bw`

Сравните уровень метилирования, покрытие, GC content и CpG observed/expected в нескольких локусах

## Вопросы и задания

1. Почему coverage track нужно смотреть вместе с треком метилирования?
2. Где в IGV видны участки с высоким GC content?
3. Совпадают ли CpG богатые регионы с низким метилированием?
4. Самостоятельно: добавьте к просмотру H3K9me3 или H3K27ac track из ChIP-seq практики и сравните с метилированием